# Training Your Own RL Agent on SupplyMind

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShAuRyA-Noodle/Sleep-Token/blob/main/notebooks/02_training_your_own_agent.ipynb)

This notebook walks through training a PPO agent on the SupplyMind environment with full hyperparameter explanations.

In [1]:
# Bootstrap so `import rl` / `from server...` resolve on Colab (git clone) and local Jupyter (sys.path).
import os, sys, subprocess
from pathlib import Path
REPO_URL = 'https://github.com/ShAuRyA-Noodle/Sleep-Token.git'
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    ROOT = Path('/content/Sleep-Token')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)
    os.chdir(ROOT)
else:
    # Local Jupyter: notebook lives in notebooks/, so the repo root is the parent dir.
    ROOT = Path.cwd()
    if not (ROOT / 'rl').exists() and (ROOT.parent / 'rl').exists():
        ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'Repo root on sys.path: {ROOT}  (Colab={IN_COLAB})')


Repo root on sys.path: c:\Users\Dell\Desktop\Sleep-Token  (Colab=False)


In [2]:
# [nb-exec-patched]
# Colab installs deps here; a local venv already has them. Guarded so a local run
# never downgrades a shared environment (e.g. pinned trl) or requires network.
try:
    import google.colab  # noqa: F401
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False
if _IN_COLAB:
    import subprocess, sys as _sys
    subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', 'gymnasium==0.29.1', 'stable-baselines3==2.2.1', 'sb3-contrib==2.2.1', 'torch', 'networkx', 'numpy', 'pydantic', 'fastapi'], check=False)
    print('install ok (Colab)')
else:
    print('Local venv detected — skipping pip install (deps already present).')


Local venv detected — skipping pip install (deps already present).


In [3]:
import gymnasium as gym
import torch
import numpy as np
import rl

from sb3_contrib import MaskablePPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


PyTorch: 2.5.1+cu121
CUDA: True


In [4]:
# [nb-exec-patched] Use the production masking wrapper (rl.train_ppo.make_env) so
# MaskablePPO gets the correct MultiDiscrete([7,40]) -> 47-dim per-dimension masks
# (the raw env exposes a 280-dim joint mask; ActionMasker + _get_action_masks convert it).
from rl.train_ppo import make_env

n_envs = 4  # 4 parallel environments (increase to 32 on GPU)
env = DummyVecEnv([make_env('SupplyMind-Easy-v1', 42, i) for i in range(n_envs)])
env = VecNormalize(env, norm_obs=True, norm_reward=True)
print(f'Vectorized env with {n_envs} parallel workers (ActionMasker-wrapped)')


Vectorized env with 4 parallel workers (ActionMasker-wrapped)


In [5]:
# Configure PPO
model = MaskablePPO(
    'MlpPolicy',
    env,
    n_steps=512,         # Steps per env before update (lower for Colab)
    batch_size=128,      # Mini-batch size for gradient updates
    learning_rate=3e-4,  # Adam learning rate
    gamma=0.99,          # Discount factor (long-term planning)
    gae_lambda=0.95,     # GAE smoothing (bias-variance tradeoff)
    clip_range=0.2,      # PPO clipping (prevents too-large updates)
    ent_coef=0.01,       # Entropy bonus (encourages exploration)
    device='cuda' if torch.cuda.is_available() else 'cpu',
    verbose=1,
)
print(f'Training on {model.device}')

Using cuda device


Training on cuda


In [6]:
# [nb-exec-patched]
# smoke: 3000 timesteps to prove the training loop runs on CPU in seconds.
# full run: model.learn(total_timesteps=2_000_000, progress_bar=True) on GPU.
model.learn(total_timesteps=3000, progress_bar=False)


-----------------------------
| time/              |      |
|    fps             | 250  |
|    iterations      | 1    |
|    time_elapsed    | 8    |
|    total_timesteps | 2048 |
-----------------------------


-----------------------------------------
| time/                   |             |
|    fps                  | 210         |
|    iterations           | 2           |
|    time_elapsed         | 19          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.009439641 |
|    clip_fraction        | 0.0606      |
|    clip_range           | 0.2         |
|    entropy_loss         | -4.39       |
|    explained_variance   | -0.22468615 |
|    learning_rate        | 0.0003      |
|    loss                 | 0.191       |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0154     |
|    value_loss           | 0.688       |
-----------------------------------------


In [7]:
# Evaluate
eval_env = gym.make('SupplyMind-Easy-v1')
rewards = []

for ep in range(10):
    obs, info = eval_env.reset(seed=ep)
    total_r = 0
    while True:
        action, _ = model.predict(obs, deterministic=True)
        obs, r, term, trunc, info = eval_env.step(action)
        total_r += r
        if term or trunc:
            break
    rewards.append(total_r)

print(f'Mean reward: {np.mean(rewards):.4f} +/- {np.std(rewards):.4f}')

Mean reward: 0.9650 +/- 0.0076
